# Fairness Audit and Ethical Analysis

**SENTINEL-CXR** — Uncertainty-Aware Chest Radiograph Triage
Deep Learning (MAIB AI 114) · Prof Anshul Gupta · S P Jain School of Global Management, Dubai

| Group member | Student ID |
|---|---|
| Krishna Mathur | AS25DXB018 |
| Atharva Soundankar | AS25DXB020 |
| Yash Petkar | AS25DXB021 |

---

**Syllabus mapping — Week 12: Ethical and Societal Impacts of Deep Learning**

Learning outcome E is assessed **only** in the final group project, so this is a
graded requirement rather than an appendix.

Performance is disaggregated across sex, age band, and view position. The last of
these is the most interesting: portable AP films are taken of patients too unwell
to stand, so view position correlates with severity. A model can learn to read
*"this is an AP film"* as *"this patient is sick"* — a shortcut that scores well on
the test set and fails the moment acquisition practice changes.

This notebook produces `fairness_report.json`, which the deployed API serves.


In [ ]:
# ── Environment ───────────────────────────────────────────────────────
# Runs on Colab free tier (T4). Nothing here needs a paid runtime.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "-q", "install",
         "torchxrayvision", "scikit-learn", "seaborn"],
        check=False,
    )

import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

SEED = 20260812
np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch {torch.__version__} | device {DEVICE}")

plt.rcParams.update({
    "figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 9, "axes.grid": True, "grid.alpha": 0.25,
})
INSTRUMENT, STAT = "#2E9CB8", "#D64541"

In [ ]:
# ── Data ──────────────────────────────────────────────────────────────
# NIH ChestX-ray14: 112,120 frontal radiographs, 30,805 patients, 14 labels.
# Kaggle: https://www.kaggle.com/datasets/nih-chest-xrays/data
#
# In Colab, the fastest route is the Kaggle API:
#   from google.colab import files; files.upload()      # kaggle.json
#   !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
#   !kaggle datasets download -d nih-chest-xrays/data -p /content/nih --unzip

DATA_DIR = os.environ.get("NIH_DIR", "/content/nih")
META = os.path.join(DATA_DIR, "Data_Entry_2017.csv")

PATHOLOGIES = ["Atelectasis","Cardiomegaly","Consolidation","Edema","Effusion",
               "Emphysema","Fibrosis","Hernia","Infiltration","Mass","Nodule",
               "Pleural_Thickening","Pneumonia","Pneumothorax"]

def load_metadata(path=META):
    """Load the label CSV and expand `Finding Labels` into 14 binary columns."""
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    for p in PATHOLOGIES:
        df[p] = df["Finding Labels"].str.contains(p, regex=False).astype(int)
    df["Patient Age"] = pd.to_numeric(df["Patient Age"], errors="coerce")
    # Ages above ~100 in this dataset are data-entry errors, not centenarians.
    df = df[(df["Patient Age"] > 0) & (df["Patient Age"] < 100)]
    return df

def patient_disjoint_split(df, fracs=(0.70, 0.10, 0.20), seed=SEED):
    """Split by Patient ID — NEVER by image.

    A patient contributes 3-4 follow-up studies. Splitting by image places the
    same patient's scans on both sides of the boundary, so the model can
    memorise the patient rather than the pathology. Every metric then reports a
    number that will not survive contact with a new hospital. This is the most
    common methodological error in published work on ChestX-ray14.
    """
    patients = df["Patient ID"].unique()
    rng = np.random.default_rng(seed)
    rng.shuffle(patients)
    n = len(patients)
    a, b = int(fracs[0] * n), int((fracs[0] + fracs[1]) * n)
    sets = (set(patients[:a]), set(patients[a:b]), set(patients[b:]))
    train, cal, test = (df[df["Patient ID"].isin(s)].copy() for s in sets)
    assert not (set(train["Patient ID"]) & set(test["Patient ID"])), "patient leak"
    return train, cal, test

## 1. Disaggregated performance

In [ ]:
from sklearn.metrics import roc_auc_score, confusion_matrix

def stratified_metrics(probs, labels, meta, pathology, threshold=0.5):
    """AUROC, TPR and FPR within each stratum."""
    k = PATHOLOGIES.index(pathology)
    meta = meta.reset_index(drop=True)
    meta["age_band"] = pd.cut(meta["Patient Age"], [0, 30, 50, 70, 100],
                              labels=["<30", "30-50", "50-70", "70+"])
    rows = []
    for stratum in ["Patient Gender", "age_band", "View Position"]:
        for value, idx in meta.groupby(stratum, observed=True).groups.items():
            idx = np.asarray(idx)
            y, p = labels[idx, k], probs[idx, k]
            if y.sum() < 20 or y.sum() == len(y):
                continue        # too few positives for a stable estimate
            tn, fp, fn, tp = confusion_matrix(y, p >= threshold, labels=[0,1]).ravel()
            rows.append({
                "stratum": stratum, "value": str(value), "n": len(idx),
                "n_positive": int(y.sum()),
                "auc": roc_auc_score(y, p),
                "tpr": tp / max(tp + fn, 1),
                "fpr": fp / max(fp + tn, 1),
            })
    return pd.DataFrame(rows)

def equalised_odds_gap(df):
    """Max within-stratum difference in TPR and FPR.

    Equalised odds asks for equal TPR *and* equal FPR across groups. Equal
    accuracy is not sufficient: a model can be equally accurate on two groups
    while missing far more disease in one of them.
    """
    out = []
    for stratum, g in df.groupby("stratum"):
        out.append({"stratum": stratum,
                    "tpr_gap": g["tpr"].max() - g["tpr"].min(),
                    "fpr_gap": g["fpr"].max() - g["fpr"].min(),
                    "auc_gap": g["auc"].max() - g["auc"].min()})
    return pd.DataFrame(out)

print("Fairness metrics ready.")

## 2. The view-position shortcut

The most important experiment in this notebook.

In [ ]:
def shortcut_probe(df):
    """Can a model predict VIEW POSITION from the image alone?

    If a classifier trained only to distinguish AP from PA reaches high AUROC,
    that signal is plainly available in the pixels — and a pathology model
    trained on the same images can exploit it as a proxy for severity, because
    AP films come from sicker, less mobile patients.

    Also report pathology prevalence by view. A large gap is direct evidence of
    the confound.
    """
    if "View Position" not in df.columns:
        print("Requires the metadata CSV."); return
    print(df.groupby("View Position")[PATHOLOGIES].mean().T
            .rename(columns=lambda c: f"prev_{c}")
            .assign(gap=lambda d: (d.iloc[:,0]-d.iloc[:,1]).abs())
            .sort_values("gap", ascending=False)
            .to_string(float_format=lambda v: f"{v:.4f}"))
    print("\nPathologies with the largest AP/PA prevalence gap are the ones")
    print("most exposed to this shortcut. Report them explicitly as a limitation.")

print("shortcut_probe ready.")

## 3. Export the audit and write the model card

In [ ]:
import json
from datetime import datetime, timezone

TOLERANCE = 0.10

def export_fairness(df, gaps, pathology, path="fairness_report.json"):
    worst = float(max(gaps["tpr_gap"].max(), gaps["fpr_gap"].max()))
    payload = {
        "generated_at": datetime.now(timezone.utc).isoformat(),
        "pathology": pathology,
        "strata": df.to_dict(orient="records"),
        "gaps": gaps.to_dict(orient="records"),
        "max_equalised_odds_gap": worst,
        "within_tolerance": bool(worst <= TOLERANCE),
        "tolerance": TOLERANCE,
        "note": ("Disparities are reported whether or not they are favourable. "
                 "A gap within tolerance is not evidence of fairness — only that "
                 "this particular audit did not detect a violation."),
    }
    json.dump(payload, open(path, "w"), indent=2)
    print(f"Wrote {path} -> copy to apps/api/artifacts/")
    print(f"max equalised-odds gap {worst:.4f} (tolerance {TOLERANCE})")
    return payload

print("""
LIMITATIONS TO STATE IN THE REPORT — all of them

1. Labels were NLP-mined from radiology reports with ~10% error. Every metric
   inherits that ceiling; no result here can be more accurate than its labels.
2. Single-institution US data. Performance on other populations, other
   equipment, and other acquisition practice is unvalidated.
3. AP/PA view position confounds severity (see the shortcut probe above).
4. Grad-CAM shows correlation, not causation. A plausible heat map is not
   evidence of correct reasoning.
5. Automation bias: a confident wrong answer is more dangerous than no answer.
   This is the direct justification for the abstention mechanism.
6. No race or ethnicity labels exist in ChestX-ray14, so a major axis of known
   medical-AI disparity CANNOT be audited here. This absence is itself a
   finding and must not be presented as an absence of bias.
""")

---

### References for this notebook

- Obermeyer, Z. et al. (2019). Dissecting racial bias in an algorithm used to manage population health. *Science*.
- Seyyed-Kalantari, L. et al. (2021). Underdiagnosis bias of AI algorithms in under-served patient populations. *Nature Medicine*.
- Hardt, M., Price, E. & Srebro, N. (2016). Equality of opportunity in supervised learning. *NeurIPS*.
- Mitchell, M. et al. (2019). Model cards for model reporting. *FAT\**.
- Oakden-Rayner, L. et al. (2020). Hidden stratification causes clinically meaningful failures. *CHIL*.

---

*SENTINEL-CXR is a student research prototype. It is not a medical device and
must not be used for clinical decisions.*
